# Tox21 Phase 3: Uni-Mol 3D Toxicity Prediction (Colab GPU)

This notebook trains Uni-Mol v1 on the Tox21 dataset using a free Colab GPU.

**Instructions:**
1. Go to `Runtime → Change runtime type → T4 GPU`
2. Run all cells
3. Download `results.zip` at the end
4. Unzip into your local `tox21_phase3/` folder

In [ ]:
# Step 1: Install dependencies
!pip install -q unimol_tools>=0.1.5 huggingface_hub rdkit torch scikit-learn pandas numpy matplotlib seaborn tqdm

In [ ]:
# Step 2: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Step 3: Patch unimol_tools to handle NaN labels in multilabel classification
# This fixes a bug where NaN values are cast to integers, destroying the masking.

import importlib
import unimol_tools

# --- Patch 1: datahub.py - keep targets as float32 ---
datahub_path = unimol_tools.data.datahub.__file__
with open(datahub_path, 'r') as f:
    content = f.read()
patched = content.replace(
    ".astype(np.int32)\n            )\n            self.data['target'] = target\n        elif self.task == 'repr':",
    ".astype(np.float32)\n            )\n            self.data['target'] = target\n        elif self.task == 'repr':"
)
with open(datahub_path, 'w') as f:
    f.write(patched)

# --- Patch 2: trainer.py - don't cast multilabel targets to long ---
trainer_path = unimol_tools.tasks.trainer.__file__
with open(trainer_path, 'r') as f:
    content = f.read()
patched = content.replace(
    "['classification', 'multiclass', 'multilabel_classification']",
    "['classification', 'multiclass']"
)
with open(trainer_path, 'w') as f:
    f.write(patched)

# --- Patch 3: loss.py - remove y_true.long() in FocalLoss ---
loss_path = unimol_tools.models.loss.__file__
with open(loss_path, 'r') as f:
    content = f.read()
patched = content.replace(
    "    y_true = y_true.long()\n    y_pred = y_pred.float()",
    "    y_pred = y_pred.float()"
)
with open(loss_path, 'w') as f:
    f.write(patched)

# --- Patch 4: metrics.py - don't cast labels to int before masking ---
metrics_path = unimol_tools.utils.metrics.__file__
with open(metrics_path, 'r') as f:
    content = f.read()
patched = content.replace(
    "label.astype(int), predict.astype(np.float32)",
    "label, predict.astype(np.float32)"
).replace(
    "label.astype(int), (predict > thre).astype(int)",
    "label, (predict > thre).astype(int)"
)
with open(metrics_path, 'w') as f:
    f.write(patched)

print("All patches applied successfully!")

# Reload modules so patches take effect
importlib.reload(unimol_tools.data.datahub)
importlib.reload(unimol_tools.tasks.trainer)
importlib.reload(unimol_tools.models.loss)
importlib.reload(unimol_tools.utils.metrics)

In [ ]:
# Step 4: Download and prepare Tox21 data
import os
import pandas as pd
import numpy as np
from rdkit import Chem

TOX21_TASKS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

# Download tox21 dataset
url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/tox21.csv.gz"
df = pd.read_csv(url, compression='gzip')

# Find SMILES column
smiles_col = [c for c in df.columns if c.lower() == 'smiles'][0]

# Validate SMILES
valid_mask = df[smiles_col].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)
df = df[valid_mask].reset_index(drop=True)

# Select columns
out = df[[smiles_col] + TOX21_TASKS].copy()
out.rename(columns={smiles_col: 'smiles'}, inplace=True)

os.makedirs('data', exist_ok=True)
out.to_csv('data/tox21_prepared.csv', index=False)
print(f"Prepared {len(out)} molecules, saved to data/tox21_prepared.csv")

In [ ]:
# Step 5: Train Uni-Mol v1 — Scaffold Split
import time
from unimol_tools import MolTrain

DATA_CSV = 'data/tox21_prepared.csv'
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-4
EARLY_STOPPING = 10

print("="*60)
print("  Training Uni-Mol v1 | Split: SCAFFOLD")
print("="*60)

t0 = time.time()
clf_scaffold = MolTrain(
    task='multilabel_classification',
    data_type='molecule',
    epochs=EPOCHS,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    early_stopping=EARLY_STOPPING,
    metrics='auc',
    split='scaffold',
    kfold=1,
    save_path='./exp/unimol_v1_scaffold',
    smiles_col='smiles',
    target_cols=TOX21_TASKS,
    model_name='unimolv1',
    remove_hs=False,
)
clf_scaffold.fit(data=DATA_CSV)
elapsed = time.time() - t0
print(f"\nScaffold split training complete in {elapsed/60:.1f} min")

In [ ]:
# Step 6: Train Uni-Mol v1 — Random Split
print("="*60)
print("  Training Uni-Mol v1 | Split: RANDOM")
print("="*60)

t0 = time.time()
clf_random = MolTrain(
    task='multilabel_classification',
    data_type='molecule',
    epochs=EPOCHS,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    early_stopping=EARLY_STOPPING,
    metrics='auc',
    split='random',
    kfold=1,
    save_path='./exp/unimol_v1_random',
    smiles_col='smiles',
    target_cols=TOX21_TASKS,
    model_name='unimolv1',
    remove_hs=False,
)
clf_random.fit(data=DATA_CSV)
elapsed = time.time() - t0
print(f"\nRandom split training complete in {elapsed/60:.1f} min")

In [ ]:
# Step 7: Evaluate both models
import joblib
from unimol_tools import MolPredict
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score

def evaluate_experiment(exp_path, data_csv, tasks):
    """Evaluate a trained Uni-Mol experiment and return per-endpoint metrics."""
    predictor = MolPredict(load_model=str(exp_path))
    df = pd.read_csv(data_csv)
    smiles_list = df['smiles'].tolist()
    preds = predictor.predict(smiles_list)
    
    results = []
    for i, task in enumerate(tasks):
        y_true = df[task].values
        y_pred = preds[:, i]
        mask = ~np.isnan(y_true)
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], y_pred[mask]
        try:
            auroc = roc_auc_score(yt, yp)
        except:
            auroc = np.nan
        try:
            auprc = average_precision_score(yt, yp)
        except:
            auprc = np.nan
        try:
            bal_acc = balanced_accuracy_score(yt, (yp > 0.5).astype(int))
        except:
            bal_acc = np.nan
        results.append({
            'endpoint': task,
            'auroc': round(auroc, 4),
            'auprc': round(auprc, 4),
            'balanced_accuracy': round(bal_acc, 4),
            'num_samples': int(mask.sum()),
            'num_positives': int(yt.sum()),
        })
    return pd.DataFrame(results)

os.makedirs('results', exist_ok=True)

# Evaluate scaffold
print("Evaluating scaffold split...")
scaffold_results = evaluate_experiment('exp/unimol_v1_scaffold', DATA_CSV, TOX21_TASKS)
scaffold_results.to_csv('results/unimol_v1_scaffold_results.csv', index=False)
print(scaffold_results.to_string(index=False))

print("\nEvaluating random split...")
random_results = evaluate_experiment('exp/unimol_v1_random', DATA_CSV, TOX21_TASKS)
random_results.to_csv('results/unimol_v1_random_results.csv', index=False)
print(random_results.to_string(index=False))

# Print mean AUROCs
print(f"\n--- Mean AUROC ---")
print(f"Scaffold: {scaffold_results['auroc'].mean():.4f}")
print(f"Random:   {random_results['auroc'].mean():.4f}")

In [ ]:
# Step 8: Build cross-phase comparison table

# Hardcoded Phase 1 & 2 results
phase12_data = {
    'endpoint': TOX21_TASKS + ['Mean'],
    'rf_random': [0.82, 0.841, 0.896, 0.818, 0.768, 0.797, 0.908, 0.776, 0.834, 0.69, 0.901, 0.804, 0.8211],
    'gatv2_random': [0.7402, 0.8322, 0.8945, 0.7962, 0.6974, 0.8160, 0.9195, 0.7849, 0.8908, 0.7474, 0.8566, 0.8382, 0.8178],
    'gatv2_scaffold': [0.643, 0.8249, 0.9123, 0.8795, 0.5693, 0.6821, 0.8506, 0.8003, 0.8209, 0.7796, 0.8985, 0.8354, 0.7914],
    'dmpnn_random': [0.7845, 0.8582, 0.8918, 0.8353, 0.7514, 0.8289, 0.9306, 0.8138, 0.9138, 0.8306, 0.8978, 0.892, 0.8524],
    'dmpnn_scaffold': [0.657, 0.8346, 0.9023, 0.894, 0.5905, 0.697, 0.8308, 0.8064, 0.8534, 0.7582, 0.8867, 0.8421, 0.7961],
}
master = pd.DataFrame(phase12_data)

# Add Uni-Mol v1 results
for split_name, results_df in [('scaffold', scaffold_results), ('random', random_results)]:
    col_name = f'unimol_v1_{split_name}'
    auroc_map = dict(zip(results_df['endpoint'], results_df['auroc']))
    master[col_name] = master['endpoint'].apply(
        lambda e: auroc_map.get(e, np.nan) if e != 'Mean' else np.nan
    )
    # Fill Mean row
    master.loc[master['endpoint'] == 'Mean', col_name] = round(
        master.loc[master['endpoint'] != 'Mean', col_name].mean(), 4
    )

master.to_csv('results/all_phases_comparison.csv', index=False)
print("Master comparison table:")
print(master.to_string(index=False))

In [ ]:
# Step 9: Generate visualizations
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

def get_model_cols(df):
    return [c for c in df.columns if c != 'endpoint']

NICE_NAMES = {
    'rf_random': 'RF (Random)',
    'gatv2_random': 'GATv2 (Random)',
    'gatv2_scaffold': 'GATv2 (Scaffold)',
    'dmpnn_random': 'D-MPNN (Random)',
    'dmpnn_scaffold': 'D-MPNN (Scaffold)',
    'unimol_v1_random': 'Uni-Mol v1 (Random)',
    'unimol_v1_scaffold': 'Uni-Mol v1 (Scaffold)',
}
COLORS = {
    'rf_random': '#636e72',
    'gatv2_random': '#0984e3',
    'gatv2_scaffold': '#74b9ff',
    'dmpnn_random': '#00b894',
    'dmpnn_scaffold': '#55efc4',
    'unimol_v1_random': '#d63031',
    'unimol_v1_scaffold': '#e17055',
}

def save(fig, name):
    fig.savefig(f'results/{name}', dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  Saved: {name}')

# --- Plot 1: Mean AUROC Bar Chart ---
mean_row = master[master['endpoint'] == 'Mean']
cols = [c for c in get_model_cols(master) if pd.notna(mean_row[c].values[0])]
vals = [float(mean_row[c].values[0]) for c in cols]
colors = [COLORS.get(c, '#b2bec3') for c in cols]
labels = [NICE_NAMES.get(c, c) for c in cols]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean AUROC', fontsize=13)
ax.set_title('Tox21 Mean AUROC: All Models & Splits', fontsize=15, fontweight='bold')
ax.set_ylim(0.7, max(vals) + 0.04)
plt.xticks(rotation=25, ha='right')
save(fig, 'all_phases_mean_auroc.png')

# --- Plot 2: Heatmap ---
data = master[master['endpoint'] != 'Mean'].copy()
heatmap_data = data.set_index('endpoint')[get_model_cols(master)].astype(float)
heatmap_data.columns = [NICE_NAMES.get(c, c) for c in heatmap_data.columns]
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0.8,
            linewidths=1, ax=ax, vmin=0.5, vmax=1.0, cbar_kws={'label': 'AUROC'})
ax.set_title('AUROC Heatmap: All Models x Endpoints', fontsize=14, fontweight='bold')
save(fig, 'all_phases_heatmap.png')

# --- Plot 3: 3D vs 2D Improvement ---
if 'unimol_v1_scaffold' in data.columns and 'dmpnn_scaffold' in data.columns:
    data['delta'] = data['unimol_v1_scaffold'].astype(float) - data['dmpnn_scaffold'].astype(float)
    data_sorted = data.sort_values('delta')
    colors_bar = ['#d63031' if d > 0 else '#0984e3' for d in data_sorted['delta']]
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(data_sorted['endpoint'], data_sorted['delta'], color=colors_bar, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    for i, (ep, d) in enumerate(zip(data_sorted['endpoint'], data_sorted['delta'])):
        ax.text(d + (0.003 if d >= 0 else -0.003), i, f'{d:+.3f}',
                va='center', ha='left' if d >= 0 else 'right', fontsize=10)
    ax.set_xlabel('AUROC Difference (Uni-Mol v1 - D-MPNN)', fontsize=12)
    ax.set_title('3D vs 2D Improvement (Scaffold Split)', fontsize=14, fontweight='bold')
    save(fig, '3d_vs_2d_improvement.png')

# --- Plot 4: Progression Chart ---
progression = []
for col, label, color in [
    ('rf_random', 'RF\n(1D Fingerprints)', '#636e72'),
    ('gatv2_scaffold', 'GATv2\n(2D Graph)', '#0984e3'),
    ('dmpnn_scaffold', 'D-MPNN\n(2D Graph)', '#00b894'),
    ('unimol_v1_scaffold', 'Uni-Mol v1\n(3D Pretrained)', '#d63031'),
]:
    if col in mean_row.columns and pd.notna(mean_row[col].values[0]):
        progression.append((label, float(mean_row[col].values[0]), color))

if len(progression) >= 2:
    labels_p, values_p, colors_p = zip(*progression)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(range(len(values_p)), values_p, 'o-', color='#2d3436', markersize=12,
            linewidth=2, zorder=5)
    for i, (l, v, c) in enumerate(progression):
        ax.scatter(i, v, s=200, color=c, zorder=6, edgecolors='white', linewidth=2)
        ax.annotate(f'{v:.4f}', (i, v), textcoords='offset points', xytext=(0, 15),
                    ha='center', fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(labels_p)))
    ax.set_xticklabels(labels_p, fontsize=11)
    ax.set_ylabel('Mean AUROC (Scaffold Split)', fontsize=13)
    ax.set_title('Model Progression: 1D -> 2D -> 3D', fontsize=15, fontweight='bold')
    ax.set_ylim(min(values_p) - 0.03, max(values_p) + 0.04)
    save(fig, 'progression_chart.png')

print("\nAll visualizations generated!")

In [ ]:
# Step 10: Package results for download
import shutil

# Also copy model files needed for local inference
shutil.make_archive('results_download', 'zip', '.', 'results')
shutil.make_archive('exp_download', 'zip', '.', 'exp')

print("\nDownload files:")
print("  results_download.zip  - CSV tables and PNG plots")
print("  exp_download.zip      - Trained model weights (for local predict.py)")
print("\nUse the Colab file browser (left panel) to download these.")
print("Then unzip into your local tox21_phase3/ folder.")